# MVP Verification Test Run

This notebook is a hands-on test harness for the Burnaby prototype MVP. It is meant to be run top-to-bottom while Claude/Codex upgrades the pipeline.

Core idea:

```text
Extractors over-generate candidates.
RAG/source discovery finds evidence.
Source repair restores context.
The deterministic verifier decides.
Uncertain but plausible rules go to review, not rejection.
GIS consumes only verified rules.
```

Default mode is safe and cheap: discovery dry-runs and artifact summaries. Optional LLM cells are off by default.

## 0. Setup

Run this first. It pins the repo paths and helper functions used by the rest of the notebook.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
from datetime import datetime

try:
    import pandas as pd
except Exception:
    pd = None

ROOT = Path('/Users/thomas/Documents/Capstone_prototype/Burnaby_prototype')
TEAM_REPO = Path('/Users/thomas/Downloads/w2025-data599-capstone-projects-green-metrics-technology-main')
PY = ROOT / '.venv' / 'bin' / 'python'

assert ROOT.exists(), ROOT
assert TEAM_REPO.exists(), TEAM_REPO
assert PY.exists(), PY

os.chdir(ROOT)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with path.open(encoding='utf-8') as f:
        return json.load(f)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def run_cmd(args, *, check=False, cwd=ROOT):
    if isinstance(args, str):
        args = args.split()
    print('$', ' '.join(map(str, args)))
    result = subprocess.run(list(map(str, args)), cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-5000:])
    if result.stderr:
        print(result.stderr[-5000:])
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed with {result.returncode}')
    return result

def show_table(rows):
    if pd is None:
        for row in rows:
            print(row)
        return rows
    df = pd.DataFrame(rows)
    display(df)
    return df

print('Notebook started:', datetime.now().isoformat(timespec='seconds'))
print('ROOT:', ROOT)
print('TEAM_REPO:', TEAM_REPO)

## 1. Confirm Burnaby Is The Correct PDF

This avoids wasting time on the wrong diagnosis. The Burnaby problem should be treated as poor rule/table extraction recall, not wrong source file.

In [ ]:
desktop_pdf = Path('/Users/thomas/Desktop/R1Small-Scale-Multi-Unit-Housing-District.pdf')
repo_pdf = ROOT / 'data/bylaws/burnaby_r1/source.pdf'

assert desktop_pdf.exists(), desktop_pdf
assert repo_pdf.exists(), repo_pdf

rows = [
    {'file': 'desktop', 'path': str(desktop_pdf), 'sha256': sha256(desktop_pdf), 'bytes': desktop_pdf.stat().st_size},
    {'file': 'repo', 'path': str(repo_pdf), 'sha256': sha256(repo_pdf), 'bytes': repo_pdf.stat().st_size},
]
show_table(rows)
print('Byte-identical:', desktop_pdf.read_bytes() == repo_pdf.read_bytes())

try:
    from pypdf import PdfReader
    for label, path in [('desktop', desktop_pdf), ('repo', repo_pdf)]:
        reader = PdfReader(str(path))
        first_text = (reader.pages[0].extract_text() or '').replace('\n', ' ')[:220]
        print(label, 'pages=', len(reader.pages), 'first_text=', first_text)
except Exception as exc:
    print('PDF page check skipped:', exc)

## 2. Read Current Results Without Rerunning Anything

This gives you a quick baseline from existing artifacts. After Claude upgrades the code, rerun this cell to compare the new output folders.

In [ ]:
def summarize_verifier_output(label, out_dir):
    out_dir = Path(out_dir)
    bench = load_json(out_dir / 'benchmark_report.json', {}) or {}
    metrics = bench.get('rule_metrics', {}) or {}
    return {
        'label': label,
        'exists': out_dir.exists(),
        'out_dir': str(out_dir),
        'candidates': len(load_json(out_dir / 'rule_candidates.json', []) or []),
        'evidence': len(load_json(out_dir / 'evidence_units.json', []) or []),
        'verified': len(load_json(out_dir / 'verified_rules.json', []) or []),
        'review': len(load_json(out_dir / 'review_needed.json', []) or []),
        'rejected': len(load_json(out_dir / 'rejected_rules.json', []) or []),
        'not_used': len(load_json(out_dir / 'not_used.json', []) or []),
        'false_verified': metrics.get('false_verified_count'),
        'precision': metrics.get('verified_precision'),
        'verified_or_review_recall': metrics.get('verified_or_review_recall'),
    }

baseline_dirs = {
    'Burnaby P5 verifier': ROOT / 'outputs/burnaby_r1_slim_pipeline5_registry',
    'Vancouver P5 verifier': ROOT / 'outputs/vancouver_rs_slim_pipeline5_registry',
    'Calgary P5 verifier': ROOT / 'outputs/calgary_rcg_slim_pipeline5_registry',
    'Burnaby P9 verifier': ROOT / 'outputs/burnaby_r1_p9',
    'Vancouver P9 verifier': ROOT / 'outputs/vancouver_rs_p9',
    'Calgary P9 verifier': ROOT / 'outputs/calgary_rcg_p9',
}

baseline_rows = [summarize_verifier_output(label, path) for label, path in baseline_dirs.items()]
baseline_df = show_table(baseline_rows)

## 3. Compare Zihao's Saved Extraction Counts

This reads the downloaded team repo. It does not treat Zihao's output as truth; it shows how many candidates his extraction generated so we can compare recall pressure.

In [ ]:
def summarize_zihao_extraction(city):
    out = TEAM_REPO / f'code/prototype_pipeline_9/outputs/{city}/06_rule_extraction_target_strict'
    summary = load_json(out / 'summary.json', {}) or {}
    return {
        'city': city,
        'exists': out.exists(),
        'text_blocks': summary.get('text_block_count'),
        'raw_text_rules': summary.get('text_rule_count'),
        'deduplicated_rules': summary.get('deduplicated_rule_count'),
        'merge_review_rules': summary.get('merge_review_rule_count'),
        'quality_flags': summary.get('extraction_quality_flags'),
        'path': str(out),
    }

zihao_rows = [summarize_zihao_extraction(city) for city in ['burnaby', 'vancouver', 'calgary']]
show_table(zihao_rows)

## 4. Dry-Run V2 Full-Bylaw Discovery

This is the cheap MVP smoke test. It builds source chunks and evidence packs without calling an LLM. Calgary should be treated as a full 1,053-page source, not a tiny slice.

Run this after any discovery/RAG/source-repair upgrade.

In [ ]:
RUN_DRY_DISCOVERY = True
DRY_CITIES = ['burnaby_r1', 'vancouver_rs', 'calgary_rcg']

if RUN_DRY_DISCOVERY:
    for city in DRY_CITIES:
        print('\n=== DRY DISCOVERY:', city, '===')
        run_cmd([
            PY, 'scripts/run_v2_bakeoff.py',
            '--city', city,
            '--dry-run',
            '--no-verify',
            '--no-examiner',
            '--max-packs', '80',
            '--max-pack-chars', '3600',
        ], check=False)
else:
    print('Dry discovery skipped')

## 5. Inspect Discovery Coverage

Important: `page_count` in the V2 summary currently means pages with extracted chunks, not total PDF pages. Use `first_page` and `last_page` to see whether the source spans the expected document range.

In [ ]:
def summarize_source(city):
    summary = load_json(ROOT / f'outputs/v2_runs/{city}/source_summary.json', {}) or {}
    lanes = {row.get('name'): row.get('count') for row in summary.get('lane_counts', [])}
    types = {row.get('name'): row.get('count') for row in summary.get('evidence_type_counts', [])}
    return {
        'city': city,
        'source_pdf': summary.get('source_pdf'),
        'source_chunks': summary.get('source_chunk_count'),
        'evidence_packs': summary.get('evidence_pack_count'),
        'pages_with_chunks': summary.get('page_count'),
        'first_page': summary.get('first_page'),
        'last_page': summary.get('last_page'),
        'lanes': lanes,
        'evidence_types': types,
    }

source_rows = [summarize_source(city) for city in DRY_CITIES]
show_table(source_rows)

## 6. Optional LLM Extraction + Verification

Leave `RUN_LLM = False` unless you are ready to spend API calls. This uses `.env` or the shell environment for `OPENROUTER_API_KEY`. Do not paste keys into this notebook.

For an MVP today, start with Burnaby and Vancouver. Calgary extraction can stay bounded after full-bylaw discovery.

In [ ]:
RUN_LLM = False
LLM_CITIES = ['burnaby_r1', 'vancouver_rs']
LLM_MODELS = 'google/gemini-2.5-flash-lite'

if RUN_LLM:
    for city in LLM_CITIES:
        print('\n=== LLM BAKEOFF:', city, '===')
        run_cmd([
            PY, 'scripts/run_v2_bakeoff.py',
            '--city', city,
            '--models', LLM_MODELS,
            '--max-packs', '80',
            '--max-pack-chars', '3600',
            '--llm-max-tokens', '1600',
            '--no-examiner',
        ], check=False)
else:
    print('LLM extraction skipped. Set RUN_LLM = True when ready.')

## 7. Summarize V2 Model Runs

This reads `outputs/v2_runs/<city>/*/` model directories and reports the verifier categories. It works whether the runs were created earlier or by this notebook.

In [ ]:
def summarize_v2_model_dirs():
    rows = []
    for city_dir in sorted((ROOT / 'outputs/v2_runs').glob('*')):
        if not city_dir.is_dir():
            continue
        for model_dir in sorted(city_dir.iterdir()):
            if not model_dir.is_dir():
                continue
            if not (model_dir / 'extraction_summary.json').exists():
                continue
            row = summarize_verifier_output(f'{city_dir.name}/{model_dir.name}', model_dir)
            summary = load_json(model_dir / 'extraction_summary.json', {}) or {}
            cost = load_json(model_dir / 'model_cost_report.json', {}) or {}
            row.update({
                'city': city_dir.name,
                'model_dir': model_dir.name,
                'dry_run': summary.get('dry_run'),
                'retrieval_packs': summary.get('retrieval_pack_count'),
                'estimated_cost_usd': cost.get('estimated_cost_usd'),
                'latency_ms': cost.get('latency_ms'),
                'extraction_errors': cost.get('extraction_error_count'),
            })
            rows.append(row)
    return rows

v2_rows = summarize_v2_model_dirs()
v2_df = show_table(v2_rows)

## 8. Review/Rejection Audit

The MVP goal is not to eliminate review. The goal is to stop hard-rejecting plausible rules that should be reviewed or marked outside scope.

Use this cell to inspect why rules were rejected.

In [ ]:
AUDIT_DIR = ROOT / 'outputs/burnaby_r1_p9'  # change to any verifier output dir

rejected = load_json(AUDIT_DIR / 'rejected_rules.json', []) or []
review = load_json(AUDIT_DIR / 'review_needed.json', []) or []
not_used = load_json(AUDIT_DIR / 'not_used.json', []) or []

def compact_rule(rule):
    return {
        'rule_object': rule.get('rule_object'),
        'operator': rule.get('operator'),
        'value': rule.get('value'),
        'unit': rule.get('unit'),
        'action': rule.get('extraction_final_action') or rule.get('decision') or rule.get('verification_status'),
        'support_gaps': rule.get('support_gaps') or rule.get('extraction_review_reasons') or rule.get('review_reasons'),
        'section': rule.get('section') or rule.get('source_section'),
        'evidence': (rule.get('evidence_text') or rule.get('source_quote') or '')[:180],
    }

print('AUDIT_DIR:', AUDIT_DIR)
print('rejected:', len(rejected), 'review:', len(review), 'not_used:', len(not_used))
show_table([compact_rule(rule) for rule in rejected[:25]])

## 9. MVP Pass/Fail Checklist

Use this at the end of a test run.

- Burnaby uses the correct 7-page R1 PDF.
- Burnaby candidate count improves compared with the weak V2 run.
- Vancouver still produces extraction/verification output.
- Calgary source discovery spans the full bylaw source, not a seven-page slice.
- `false_verified_count` stays `0` for benchmarked outputs.
- Plausible-but-incomplete rules go to `Needs review`, not hard rejection.
- Out-of-GIS-scope rules go to `Not used`, not failed numeric verification.
- Dashboard categories are plain language: `Verified`, `Needs review`, `Not used`, `Rejected`.
- GIS consumes only `verified_rules.json` / `gis_rule_contract.json`.

If one of these fails, write down the exact output directory and the file that proves it.